# MDR-TD v0
**Persistence Baseline for Soil Moisture Prediction (ΔSM)**

**Author:** Yicheng (Roy) Pan  
**Affiliation:** Seattle University, Computer Science  
**Project:** MDR  
**Notebook Type:** Baseline Evaluation (No Training)  
**Last Updated:** May 29, 2026

---

## Model Summary
- **Model Name:** MDR-TD
- **Version:** v0
- **Task:** Regression (Soil Moisture at 5 cm depth)  
- **Target Variable:** delta of `soil_moisture_5cm`  
- **Temporal Resolution:** Daily  

---

## Baseline Strategy
This notebook establishes the **persistence baseline** — the simplest possible forecast:
- **No model training, no learned inference**
- Prediction: ΔSM = 0 (no change from yesterday)
- Reconstructed SM = `soil_moisture_5cm_lag1` (yesterday's value)
- All future ML models must outperform this naive baseline

---

## Reproducibility
- **Random Seed:** 42
- **Split Metadata:** `data/splits/derived_8.0/split_meta.json`
- **Environment:** Jupyter Lab + VS Code

In [3]:
import os
import sys
import random
from pathlib import Path
import time
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import torch


PROJECT_ROOT = os.path.abspath("../../../")
print(f"project_root: {PROJECT_ROOT}")
if PROJECT_ROOT not in sys.path:
    # sys.path.insert(0, PROJECT_ROOT)
    sys.path.append(PROJECT_ROOT)

try:
    from Models.Utils.dashboard import metrics_dashboard
    print("Imported dashboard from Models.Utils")
except ImportError as e:
    from Utils.dashboard import metrics_dashboard
    print("Imported dashboard from local Utils")

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("imports loaded")
print(f"torch using: {device}")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

SEED = 42
DEEP_SEARCH = 40

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def _gpu_available():
    try:
        import subprocess
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            capture_output=True, text=True, timeout=5
        )
        return result.returncode == 0 and bool(result.stdout.strip())
    except Exception:
        return False

HAS_GPU = _gpu_available()

print(f"Random seed set to {SEED}")

def print_env_info():
    print("Environment information:")
    print(f"  Python version: {os.sys.version.split()[0]}")
    print(f"  NumPy version:  {np.__version__}")
    print(f"  Pandas version: {pd.__version__}")

    try:
        import xgboost
        print(f"  XGBoost version: {xgboost.__version__}")
    except ImportError:
        print("  XGBoost not installed")

    IN_COLAB = "COLAB_GPU" in os.environ
    print(f"  Running in Colab: {IN_COLAB}")

    if IN_COLAB:
        gpu = os.environ.get("COLAB_GPU", None)
        print(f"  GPU available: {gpu}")
    else:
        print(f"  GPU available: {HAS_GPU}")

print_env_info()

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True

print("environment setup complete")

project_root: C:\Users\pan\Documents\GitHub\MDR-Project
Imported dashboard from Models.Utils
imports loaded
torch using: cpu
Random seed set to 42
Environment information:
  Python version: 3.12.8
  NumPy version:  2.4.4
  Pandas version: 3.0.3
  XGBoost version: 3.2.0
  Running in Colab: False
  GPU available: True
environment setup complete


In [4]:
VERSION    = "v0"
SUBVERSION = "v0"
RUN_NAME   = "mdr_td_v0"

DATA_ROOT    = f"{PROJECT_ROOT}/Temporal/Pipeline/data"
SPLIT_ROOT   = f"{DATA_ROOT}/splits"
OUTPUT_ROOT  = f"{PROJECT_ROOT}/Models/TemporalDelta/{VERSION}/{SUBVERSION}"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print("Project paths:")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_ROOT:    {DATA_ROOT}")
print(f"  SPLIT_ROOT:   {SPLIT_ROOT}")
print(f"  OUTPUT_ROOT:  {OUTPUT_ROOT}")

print("\nKey file checks:")
print("  data exists:",   os.path.exists(DATA_ROOT))
print("  splits exists:", os.path.exists(SPLIT_ROOT))
print("  output exists:", os.path.exists(OUTPUT_ROOT))

Project paths:
  PROJECT_ROOT: C:\Users\pan\Documents\GitHub\MDR-Project
  DATA_ROOT:    C:\Users\pan\Documents\GitHub\MDR-Project/Temporal/Pipeline/data
  SPLIT_ROOT:   C:\Users\pan\Documents\GitHub\MDR-Project/Temporal/Pipeline/data/splits
  OUTPUT_ROOT:  C:\Users\pan\Documents\GitHub\MDR-Project/Models/TemporalDelta/v0/v0

Key file checks:
  data exists: True
  splits exists: True
  output exists: True


In [5]:
TRAIN_PATH = str(Path(SPLIT_ROOT) / "derived_8.0/train.csv")
VAL_PATH   = str(Path(SPLIT_ROOT) / "derived_8.0/val.csv")
TEST_PATH  = str(Path(SPLIT_ROOT) / "derived_8.0/test.csv")

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing split file: {p}")

print("Split files:")
print(" ", TRAIN_PATH)
print(" ", VAL_PATH)
print(" ", TEST_PATH)

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

Split files:
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_8.0\train.csv
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_8.0\val.csv
  C:\Users\pan\Documents\GitHub\MDR-Project\Temporal\Pipeline\data\splits\derived_8.0\test.csv


## Features

In [6]:
TARGET_COL = "soil_moisture_5cm"
TARGET_DELTA_COL = "delta_soil_moisture_5cm"
Y_LAG1_COL = "soil_moisture_5cm_lag1"
KEEP_META_COLS = ["station_id", "date", "longitude", "latitude"]

FEATURE_COLS = [
    'SMAP_sm_pm_interp_ema02',
    'V_rollmin_LST_modis_kobs30',
    'D_sin_DOY', 'G_rain_sum_3d',
    'V_rollmin_G_API_kobs30',
    'G_rain_sum_7d',
    'C_lag_LST_modis_kobs30',
    'C_lag_G_API_kobs1',
    'V_ema_G_API_kobs14',
    'V_rollmean_G_API_kobs14',
    'G_API', 'G_DSLR',
    'SMAP_ampm_diff_interp',
    'V_rollmax_G_API_kobs30',
    'V_ema_G_API_kobs30',
    'V_rollmean_s2_b11_kobs7',
    'V_ema_LST_modis_kobs7',
    'V_rollmean_G_API_kobs7',
    'C_lag_s2_b11_kobs30',
    'A_d_E_SAR_diff_kobs14',
    'C_lag_LST_modis_kobs6',
    'A_d_LST_modis_kobs14',
    'A_d_SMAP_sm_interp_kobs14',
    'V_rollstd_SMAP_sm_interp_kobs30',
    'SMAP_sm_interp_grad7',
    'year_frac', 'sin_year', 'cos_year',
    'API_x_year', 'SMAP_x_year',
    'slope', 'elev', 'K_slope_sin',
    'K_slope_cos', 'K_aspect_cos',
    'J_clay_wfrac_b0', 'J_sand_wfrac_b0',
    'soil_moisture_5cm_lag1'
]

In [7]:
# Create delta target (day-over-day first difference)
for df in [train_df, val_df, test_df]:
    df["date"] = pd.to_datetime(df["date"])
    df.sort_values(["station_id", "date"], inplace=True)
    df[Y_LAG1_COL] = df.groupby("station_id")[TARGET_COL].shift(1)
    df[TARGET_DELTA_COL] = df[TARGET_COL] - df[Y_LAG1_COL]
    df.dropna(subset=[TARGET_DELTA_COL], inplace=True)
    df.reset_index(drop=True, inplace=True)

print(f"train rows after delta: {len(train_df)}")
print(f"val   rows after delta: {len(val_df)}")
print(f"test  rows after delta: {len(test_df)}")

train rows after delta: 6863
val   rows after delta: 2716
test  rows after delta: 4011


## Utils

In [8]:
def get_metrics_dict(y_true, y_pred, prefix=""):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    err = y_true - y_pred
    ae = np.abs(err)

    r2 = float(r2_score(y_true, y_pred))
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(root_mean_squared_error(y_true, y_pred))

    bias = float(np.mean(err))
    ubrmse = float(np.std(err))

    q_err = np.quantile(err, [0.05, 0.25, 0.50, 0.75, 0.95])

    return {
        f"{prefix}n": int(len(y_true)),
        f"{prefix}r2": r2,
        f"{prefix}mae": mae,
        f"{prefix}rmse": rmse,
        f"{prefix}ubrmse": ubrmse,
        f"{prefix}bias": bias,
        f"{prefix}med_ae": float(np.median(ae)),
        f"{prefix}p90_ae": float(np.quantile(ae, 0.90)),
        f"{prefix}q05_err": float(q_err[0]),
        f"{prefix}q50_err": float(q_err[2]),
        f"{prefix}q95_err": float(q_err[4]),
    }

def neat_print(metrics, title=None, decimals=5):
    preferred_order = [
        "n", "r2", "mae", "rmse", "ubrmse", "bias",
        "med_ae", "p90_ae",
        "q05_err", "q50_err", "q95_err",
    ]

    keys = list(metrics.keys())
    if "_" in keys[0]:
        prefix = keys[0].split("_")[0] + "_"
    else:
        prefix = ""

    ordered_keys = []
    for k in preferred_order:
        full_key = prefix + k
        if full_key in metrics:
            ordered_keys.append(full_key)

    for k in metrics:
        if k not in ordered_keys:
            ordered_keys.append(k)

    if title:
        print(f"\n{title}")
    print("=" * 42)
    print(f"{'Metric':<18} {'Value':>18}")
    print("-" * 42)

    for k in ordered_keys:
        v = metrics[k]

        if isinstance(v, int):
            val_str = f"{v:,}"
        else:
            val_str = f"{v:.{decimals}f}"

        print(f"{k:<18} {val_str:>18}")

    print("=" * 42)

def get_importance(model, feature_cols):
    imp = pd.Series(model.feature_importances_, index=feature_cols)
    s = imp.sum()
    if s > 0:
        imp = imp / s
    return imp

TOP_K = 15

def plot_top_features(model, feature_cols, title):
    imp = get_importance(model, feature_cols)
    imp_top = imp.sort_values(ascending=False).head(TOP_K)

    print(f"\n{title} - Top {TOP_K} Features")
    print("=" * 50)
    print(imp_top.to_string())

    plt.figure(figsize=(6, max(4, 0.4 * TOP_K)))
    y = np.arange(len(imp_top))

    plt.barh(y, imp_top.values)
    plt.yticks(y, imp_top.index)
    plt.gca().invert_yaxis()

    plt.title(title)
    plt.xlabel("Normalized importance")
    plt.grid(axis="x", alpha=0.3)

    plt.tight_layout()
    plt.show()

### Split

In [9]:
print("=== SPLIT SUMMARY ===")

def split_summary(name, d):
    print(f"\n{name.upper()}")
    print(f"  rows:     {len(d)}")
    print(f"  stations: {sorted(d['station_id'].unique().tolist())}")
    if "date" in d.columns:
        print(f"  date range: {d['date'].min()} -- {d['date'].max()}")

split_summary("train", train_df)
split_summary("val", val_df)
split_summary("test", test_df)

print("\n=== LEAKAGE CHECK ===")
print("train \u2229 test:", sorted(set(train_df.station_id) & set(test_df.station_id)))
print("val   \u2229 test:", sorted(set(val_df.station_id) & set(test_df.station_id)))

print("\n-- split locked --")

=== SPLIT SUMMARY ===

TRAIN
  rows:     6863
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2017-01-02 00:00:00 -- 2020-12-31 00:00:00

VAL
  rows:     2716
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']
  date range: 2021-01-02 00:00:00 -- 2022-12-31 00:00:00

TEST
  rows:     4011
  stations: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
  date range: 2023-01-02 00:00:00 -- 2025-12-31 00:00:00

=== LEAKAGE CHECK ===
train ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane', 'Touchet_WA_824']
val   ∩ test: ['Darrington', 'Quinault', 'SourdoughGulch_WA_985', 'Spokane']

-- split locked --


## Baseline: Persistence Model (No Training)

Naive baseline: predict ΔSM = 0 (no change from yesterday).
Reconstructed SM = `soil_moisture_5cm_lag1` (yesterday's value).

In [10]:
# --- Persistence baseline: predict delta = 0 ---
y_val_delta_pred = np.zeros(len(val_df))
y_test_delta_pred = np.zeros(len(test_df))

# --- Delta metrics (actual vs 0) ---
val_delta_metrics = get_metrics_dict(val_df[TARGET_DELTA_COL], y_val_delta_pred, prefix="val_delta_")
neat_print(val_delta_metrics, title="VAL \u2014 Delta SM (baseline \u0394=0)")
metrics_dashboard(val_df[TARGET_DELTA_COL], y_val_delta_pred, name="Val \u0394SM (baseline)")

test_delta_metrics = get_metrics_dict(test_df[TARGET_DELTA_COL], y_test_delta_pred, prefix="test_delta_")
neat_print(test_delta_metrics, title="TEST \u2014 Delta SM (baseline \u0394=0)")
metrics_dashboard(test_df[TARGET_DELTA_COL], y_test_delta_pred, name="Test \u0394SM (baseline)")

# --- Reconstruct raw SM from lag1 (no delta) ---
y_val_recon = val_df[Y_LAG1_COL].to_numpy().ravel()
y_test_recon = test_df[Y_LAG1_COL].to_numpy().ravel()
y_val_true = val_df[TARGET_COL].to_numpy().ravel()
y_test_true = test_df[TARGET_COL].to_numpy().ravel()

val_recon_metrics = get_metrics_dict(y_val_true, y_val_recon, prefix="val_recon_")
neat_print(val_recon_metrics, title="VAL \u2014 Reconstructed SM (lag1 = yesterday)")
metrics_dashboard(y_val_true, y_val_recon, name="Val Reconstructed SM (baseline)")

test_recon_metrics = get_metrics_dict(y_test_true, y_test_recon, prefix="test_recon_")
neat_print(test_recon_metrics, title="TEST \u2014 Reconstructed SM (lag1 = yesterday)")
metrics_dashboard(y_test_true, y_test_recon, name="Test Reconstructed SM (baseline)")


VAL — Delta SM (baseline Δ=0)
Metric                          Value
------------------------------------------
val_delta_n                     2,716
val_delta_r2                 -0.00002
val_delta_mae                 0.00898
val_delta_rmse                0.01479
val_delta_ubrmse              0.01479
val_delta_bias               -0.00007
val_delta_med_ae              0.00500
val_delta_p90_ae              0.02300
val_delta_q05_err            -0.01800
val_delta_q50_err            -0.00200
val_delta_q95_err             0.02800



TEST — Delta SM (baseline Δ=0)
Metric                          Value
------------------------------------------
test_delta_n                    4,011
test_delta_r2                -0.00001
test_delta_mae                0.00904
test_delta_rmse               0.01590
test_delta_ubrmse             0.01590
test_delta_bias              -0.00004
test_delta_med_ae             0.00500
test_delta_p90_ae             0.02200
test_delta_q05_err           -0.01800
test_delta_q50_err           -0.00200
test_delta_q95_err            0.02800



VAL — Reconstructed SM (lag1 = yesterday)
Metric                          Value
------------------------------------------
val_recon_n                     2,716
val_recon_r2                  0.98020
val_recon_mae                 0.00898
val_recon_rmse                0.01479
val_recon_ubrmse              0.01479
val_recon_bias               -0.00007
val_recon_med_ae              0.00500
val_recon_p90_ae              0.02300
val_recon_q05_err            -0.01800
val_recon_q50_err            -0.00200
val_recon_q95_err             0.02800



TEST — Reconstructed SM (lag1 = yesterday)
Metric                          Value
------------------------------------------
test_recon_n                    4,011
test_recon_r2                 0.97148
test_recon_mae                0.00904
test_recon_rmse               0.01590
test_recon_ubrmse             0.01590
test_recon_bias              -0.00004
test_recon_med_ae             0.00500
test_recon_p90_ae             0.02200
test_recon_q05_err           -0.01800
test_recon_q50_err           -0.00200
test_recon_q95_err            0.02800
